[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S08/S08_03_boosting_xgboost_lightgbm.ipynb)

# S08 · 03 — Boosting: Gradient Boosting, XGBoost y LightGBM

**Tiempo estimado:** 65 min · **Sprint 8** · M3 (UD 3.7)

**Contexto TurisData.** El bosque promedia árboles independientes. El **boosting** hace algo distinto:
entrena árboles **pequeños y en cadena**, cada uno dedicado a **corregir los errores del anterior**.
Es la familia que suele ganar en datos tabulares como los de TurisData. Probamos la versión de scikit-learn y
las dos librerías estrella: **XGBoost** y **LightGBM**.

**Al terminar sabrás:**
1. Explicar la diferencia entre *bagging* (bosque) y *boosting*.
2. Controlar el sobreajuste con `learning_rate`, número de árboles y **parada temprana** (*early stopping*).
3. Entrenar XGBoost y LightGBM y leer sus principales hiperparámetros.
4. Comparar rendimiento y tiempo de las tres implementaciones.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore")
SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["temporada_alta"] = df["fecha_llegada"].dt.month.isin([1, 2, 3, 7, 8, 12]).astype(int)
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)

prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), columnas_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), columnas_cat),
])
Xtr = prep.fit_transform(X_train)
Xte = prep.transform(X_test)
nombres = prep.get_feature_names_out()

# Para la parada temprana necesitamos un conjunto de VALIDACIÓN aparte (sacado de train, no del test)
Xfit, Xval, yfit, yval = train_test_split(Xtr, y_train, test_size=0.2, random_state=SEMILLA, stratify=y_train)
print("Entrenar:", len(Xfit), "· validar:", len(Xval), "· probar (test, intacto):", len(Xte))

## 1. La idea del boosting

1. El primer árbol (muy pequeño) hace una predicción básica.
2. Se miran los **errores** que comete (los «residuos»).
3. El segundo árbol se entrena para **predecir esos errores** y se suma al primero, multiplicado por
   un paso pequeño: el **`learning_rate`** (tasa de aprendizaje).
4. Se repite cientos de veces.

| | Bosque (*bagging*) | Boosting |
|---|---|---|
| Árboles | Grandes, independientes, en paralelo | Pequeños, en cadena, cada uno corrige al anterior |
| Reduce sobre todo | Varianza | Sesgo (y, con cuidado, también varianza) |
| Riesgo | Poco sobreajuste | **Sobreajusta si se añaden demasiados árboles** |
| Ajuste | Poco sensible | Sensible: `learning_rate`, nº de árboles, profundidad |

## 2. Sobreajuste por exceso de árboles

Con `GradientBoostingClassifier` podemos ver cómo evoluciona el AUC **de validación** a medida que
se añaden árboles (`staged_predict_proba`). Usamos una tasa de aprendizaje alta (0,2) para ver
el efecto con claridad.

In [ ]:
gb = GradientBoostingClassifier(n_estimators=300, learning_rate=0.2, max_depth=3, random_state=SEMILLA).fit(Xfit, yfit)
auc_val_por_arbol = [roc_auc_score(yval, p[:, 1]) for p in gb.staged_predict_proba(Xval)]
auc_fit_por_arbol = [roc_auc_score(yfit, p[:, 1]) for p in gb.staged_predict_proba(Xfit)]
plt.figure(figsize=(8, 4))
plt.plot(range(1, 301), auc_fit_por_arbol, label="Entrenamiento")
plt.plot(range(1, 301), auc_val_por_arbol, label="Validación")
plt.title("Gradient Boosting: AUC según el número de árboles")
plt.xlabel("Número de árboles")
plt.ylabel("AUC")
plt.legend()
plt.show()

**Ejercicio 1.** Con la lista `auc_val_por_arbol`, guarda en `mejor_n_arboles` el número de árboles
(empezando a contar en 1) con mejor AUC de validación, y en `caida_final` cuánto ha caído el AUC de
validación con los 300 árboles respecto a ese máximo (valor positivo).

In [ ]:
# TODO: np.argmax(...) + 1 para el nº de árboles; max(...) - último valor para la caída
...
assert 1 <= mejor_n_arboles < 100, "Con learning_rate 0,2 el óptimo llega pronto (pocas decenas de árboles)"
assert caida_final > 0.02, "Con 300 árboles el AUC de validación debería haber caído: sobreajuste"
print(f"Mejor nº de árboles: {mejor_n_arboles} · pérdida de AUC si seguimos hasta 300: {caida_final:.3f}")

**Solución práctica: parada temprana.** En lugar de adivinar cuántos árboles, se pone un máximo alto y se
**detiene el entrenamiento cuando la validación deja de mejorar** durante N rondas seguidas.
Además, una tasa de aprendizaje **más baja** (0,03–0,1) y más árboles suele dar mejores modelos.

## 3. XGBoost

XGBoost es una implementación muy optimizada del boosting con **regularización** incorporada.
Hiperparámetros principales:

| Hiperparámetro | Qué controla |
|---|---|
| `n_estimators` | Máximo de árboles (con parada temprana, se pone alto) |
| `learning_rate` | Paso de cada árbol (menor = más lento pero más fino) |
| `max_depth` | Profundidad de cada árbol (2–6 suele bastar) |
| `subsample` / `colsample_bytree` | Fracción de filas / columnas por árbol (reducen la varianza) |
| `reg_lambda`, `min_child_weight` | Regularización L2 y tamaño mínimo de nodo |
| `early_stopping_rounds` | Rondas sin mejorar para detenerse |

**Ejercicio 2.** Entrena `modelo_xgb = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=3,
subsample=0.8, colsample_bytree=0.8, early_stopping_rounds=30, eval_metric="auc", random_state=SEMILLA, n_jobs=2)`
con `Xfit, yfit` y `eval_set=[(Xval, yval)]` (usa `verbose=False`). Guarda `arboles_xgb`
(`modelo_xgb.best_iteration`) y `auc_test_xgb` (AUC en test).

In [ ]:
# TODO: modelo_xgb.fit(Xfit, yfit, eval_set=[(Xval, yval)], verbose=False)
...
assert 5 < arboles_xgb < 1000, "La parada temprana debería detener el entrenamiento antes del máximo"
assert auc_test_xgb > 0.72, "El AUC de XGBoost en test debería superar 0,72"
print(f"XGBoost paró en {arboles_xgb} árboles · AUC test = {auc_test_xgb:.3f}")

## 4. LightGBM

LightGBM (de Microsoft) es otra implementación muy rápida. Diferencias de estilo: hace crecer los árboles
**por hojas** (*leaf-wise*), así que el control principal es **`num_leaves`** en lugar de la profundidad;
`min_child_samples` (mínimo de reservas por hoja) evita hojas diminutas. La parada temprana se
indica con un *callback*.

**Ejercicio 3.** Entrena `modelo_lgb = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=8,
subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEMILLA, n_jobs=2, verbose=-1)` con
`Xfit, yfit`, `eval_set=[(Xval, yval)]`, `eval_metric="auc"` y `callbacks=[lgb.early_stopping(30, verbose=False)]`.
Guarda `arboles_lgb` (`modelo_lgb.best_iteration_`) y `auc_test_lgb`.

In [ ]:
# TODO: modelo_lgb.fit(Xfit, yfit, eval_set=[(Xval, yval)], eval_metric="auc", callbacks=[...])
...
assert 5 < arboles_lgb < 1000, "La parada temprana debería detener el entrenamiento antes del máximo"
assert auc_test_lgb > 0.72, "El AUC de LightGBM en test debería superar 0,72"
print(f"LightGBM paró en {arboles_lgb} árboles · AUC test = {auc_test_lgb:.3f}")

## 5. Qué mira el modelo

XGBoost y LightGBM también dan importancias de variable (por «ganancia»). Como siempre: describen el
**modelo**, no la causalidad.

In [ ]:
imp = pd.Series(modelo_xgb.feature_importances_, index=nombres).sort_values(ascending=False).head(8)
plt.figure(figsize=(7, 4))
imp.iloc[::-1].plot(kind="barh", color="#2a6f97")
plt.title("XGBoost: importancia de variables (top 8)")
plt.xlabel("Importancia (ganancia relativa)")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 6. Mini-reto: comparativa de las tres implementaciones

Crea `tabla`, un DataFrame con filas `"sklearn_gb"`, `"xgboost"` y `"lightgbm"` y columnas `auc_test`,
`arboles` y `segundos_ajuste` (tiempo de `fit` medido con `time.perf_counter()`). Para
`sklearn_gb` usa `GradientBoostingClassifier(learning_rate=0.05, max_depth=3, subsample=0.8,
n_estimators=1000, n_iter_no_change=30, validation_fraction=0.2, random_state=SEMILLA)` sobre **todo** `Xtr`
(esta versión también sabe parar antes: `n_estimators_` dice cuántos árboles usó). Para XGBoost y LightGBM
reentrénalos con los mismos parámetros de los ejercicios (con `Xfit`/`Xval`) midiendo el tiempo.

In [ ]:
# TODO: mide t0 = time.perf_counter() antes de cada .fit y resta después; n_estimators_ en sklearn
...
assert list(tabla.index) == ["sklearn_gb", "xgboost", "lightgbm"], "Filas en el orden pedido"
assert {"auc_test", "arboles", "segundos_ajuste"} <= set(tabla.columns)
assert tabla["auc_test"].between(0.70, 0.80).all(), "Los tres deberían tener AUC entre 0,70 y 0,80"
tabla

Las tres versiones llegan a un AUC muy parecido: **la implementación importa menos que los datos y la
validación**. Las diferencias reales suelen estar en la **velocidad** con muchos datos (LightGBM y
XGBoost escalan mucho mejor que la versión de scikit-learn) y en el **manejo nativo** de nulos y
categóricas (LightGBM y XGBoost lo admiten; aquí lo hemos evitado con el mismo preprocesado para poder comparar).

## 7. Para reflexionar

1. ¿Por qué el boosting puede sobreajustar por añadir árboles y el bosque prácticamente no?
2. ¿Qué haces si el AUC de validación sube durante 400 árboles y no se detiene? ¿Y si sube 20 y luego baja?
3. Reducir `learning_rate` de 0,2 a 0,05 exige más árboles. ¿Qué compensas y qué pagas?
4. Si los tres tienen AUC 0,745 ± 0,01, ¿en qué te fijarías para elegir uno?

## 8. Resumen: qué has aprendido
- Boosting = árboles pequeños en cadena que corrigen errores; **learning_rate** y nº de árboles se equilibran.
- **Parada temprana** con un conjunto de validación (nunca el test) evita el sobreajuste y ahorra tiempo.
- XGBoost y LightGBM comparten ideas y difieren en detalles (`max_depth` vs `num_leaves`).
- Lo que decide la elección suele ser el **coste** (tiempo, mantenimiento, explicabilidad), no unas centésimas de AUC.